# p155 Min Stack 解析笔记

- 题号：p155
- 题目英文名：Min Stack
- 题目中文名：最小栈
- 题目链接：https://leetcode.cn/problems/min-stack/
- 题型：栈设计
- 难度：Medium
- 推荐优先级：高
- Java 原代码位置：`solutions-java/leetcode/p155_min_stack/MinStack.java`


## 1. 题目一句话总结

这道题要求我们设计一个栈，在支持普通 `push/pop/top` 的同时，还要让 `getMin` 始终保持 `O(1)`。

本质上考的是如何在栈的每一步操作中，把“当前最小值”同步维护起来。


## 2. 题目理解与约束分析

### 2.1 题目要求

实现一个支持以下操作的栈：

- `push(x)`：压入元素
- `pop()`：弹出栈顶元素
- `top()`：返回栈顶元素
- `getMin()`：返回当前栈内最小值

其中 `getMin()` 必须做到常数时间。

### 2.2 输入与输出

- 输入：一系列栈操作
- 输出：对应操作结果，尤其是 `top()` 和 `getMin()`
- 返回结果含义：栈顶值和最小值都要与当前栈状态一致

### 2.3 关键约束

- 不能每次 `getMin` 都线性扫描
- 每次操作都要保持 `O(1)`
- 栈可能包含重复最小值

### 2.4 示例分析

假设依次执行：

```text
push(-2)
push(0)
push(-3)
getMin() -> -3
pop()
top() -> 0
getMin() -> -2
```

说明我们不仅要知道当前数据栈内容，还要同步知道每一步时的最小值。


## 3. Java 原代码

```java
package leetcode.p155_min_stack;

import java.util.Stack;

public class MinStack {

    private Stack<Integer> dataStack = new Stack<>();
    private Stack<Integer> minStack = new Stack<>();

    public MinStack() {
    }

    public void push(int val) {
        dataStack.push(val);
        if (minStack.isEmpty() || val <= minStack.peek()) {
            minStack.push(val);
        } else {
            minStack.push(minStack.peek());
        }
    }

    public void pop() {
        dataStack.pop();
        minStack.pop();
    }

    public int top() {
        return dataStack.peek();
    }

    public int getMin() {
        return minStack.peek();
    }
}
```


## 4. 先从 Java 原方案理解

Java 原方案很直接，核心就是维护两个同步栈：

- `dataStack`：正常存数据
- `minStack`：同步记录“到当前位置为止的最小值”

最关键的细节在 `push`：

- 如果当前值更小，就把当前值压入 `minStack`
- 否则，不是啥都不做，而是把 `minStack.peek()` 再压一遍

这样一来，`minStack` 和 `dataStack` 的长度始终一致，任意时刻 `minStack` 栈顶都正好对应 `dataStack` 当前状态下的最小值。


## 5. 从朴素思路到优化思路

### 5.1 最直接的想法

最直观但错误的思路是：正常用一个栈存数据，调用 `getMin()` 时遍历整个栈求最小值。

### 5.2 为什么不够好

- `getMin()` 会退化成 `O(n)`
- 题目明确要求常数时间

### 5.3 优化方向

既然不能临时扫描，那就必须在每一次 `push/pop` 时，顺手把最小值信息维护好。Java 原方案就是把“当前最小值”单独也做成一个栈。


## 6. 核心算法讲解

### 6.1 算法名称

- 双栈最小值维护
- 辅助栈设计

### 6.2 为什么想到这个算法

因为最小值不是只和栈顶有关，而是和整个栈历史有关。所以我们要把每一步的“当前最小值”也记录下来。

### 6.3 关键状态

- `data_stack`：真实数据栈
- `min_stack`：当前位置对应的最小值栈

### 6.4 步骤拆解

1. `push(val)` 时把 `val` 压入数据栈
2. 同时把“新状态下的最小值”压入最小栈
3. `pop()` 时两个栈同步弹出
4. `top()` 直接看数据栈顶
5. `getMin()` 直接看最小栈顶


## 7. 过程演示

依次执行：`push(3), push(1), push(2), push(1)`

```text
初始：
data = []
min  = []

push(3)：
data = [3]
min  = [3]

push(1)：
data = [3, 1]
min  = [3, 1]

push(2)：
data = [3, 1, 2]
min  = [3, 1, 1]

push(1)：
data = [3, 1, 2, 1]
min  = [3, 1, 1, 1]
```

这样无论当前栈顶是什么，`min[-1]` 始终就是当前最小值。


In [ ]:
class MinStack:
    def __init__(self) -> None:
        self.data_stack: list[int] = []
        self.min_stack: list[int] = []

    def push(self, val: int) -> None:
        self.data_stack.append(val)
        if not self.min_stack or val <= self.min_stack[-1]:
            self.min_stack.append(val)
        else:
            self.min_stack.append(self.min_stack[-1])

    def pop(self) -> None:
        self.data_stack.pop()
        self.min_stack.pop()

    def top(self) -> int:
        return self.data_stack[-1]

    def getMin(self) -> int:
        return self.min_stack[-1]


## 8. 代码逐段讲解

### 8.1 初始化部分

Python 里直接用两个列表模拟 Java 的两个 `Stack<Integer>`。

### 8.2 `push`

`push` 的难点不在数据栈，而在最小栈。Java 原解用的是“补齐法”：即使当前值不是新最小值，也要把旧最小值再压一遍。

### 8.3 `pop`

因为两个栈始终等长，所以弹出时必须同步弹出。

### 8.4 查询操作

- `top()`：看 `data_stack[-1]`
- `getMin()`：看 `min_stack[-1]`

所以两个查询都能保持 `O(1)`。


## 9. 复杂度分析

- 时间复杂度：`push/pop/top/getMin` 都是 `O(1)`
- 为什么是这个时间复杂度：每次只操作栈顶，不做遍历
- 空间复杂度：`O(n)`
- 为什么是这个空间复杂度：辅助最小栈与数据栈等长


## 10. 易错点

1. 只在出现新最小值时才向 `min_stack` 压入元素，导致两个栈长度不一致。
2. `pop()` 时只弹数据栈，不弹最小栈。
3. 忽略重复最小值，比如两个 `1`，弹出一个后最小值仍然应该是 `1`。
4. 把 `getMin()` 写成临时扫描，违背题目要求。


## 11. 面试时怎么讲

可以这样概括：

> 这题我会用两个栈做，一个存真实数据，一个存每一步对应的最小值。
> 关键点是 `minStack` 不只是存“新最小值”，而是每次 `push` 都压一个当前最小值进去。
> 这样两个栈长度一致，弹出时同步弹出，`getMin()` 直接看辅助栈栈顶即可。
> 所有操作都能做到 `O(1)`。


## 12. 实际应用场景

这题可以理解成：在维护一个可回退的数据栈时，同时维护一个可回退的统计量。

### 具体业务案例：价格窗口最小值回退栈

#### 业务背景

某些系统会保存一段可撤销的操作历史，每一步都可能需要知道当前最低价格或最低阈值。

#### 输入是什么

输入是一系列入栈、出栈操作，对应价格或阈值变化。

#### 算法介入点

系统不仅要维护当前栈顶状态，还要随时知道到当前为止的最小值。

#### 输出是什么

输出是当前栈顶值和当前最小值。

#### 价值是什么

它解决的是“状态回退 + 聚合信息同步维护”的问题。


In [ ]:
stack = MinStack()
stack.push(-2)
stack.push(0)
stack.push(-3)
print('当前最小值:', stack.getMin())
stack.pop()
print('当前栈顶:', stack.top())
print('弹出后的最小值:', stack.getMin())


## 13. Demo 输出说明

- 第一行应输出 `-3`，说明最小值维护正确。
- 弹出后栈顶应是 `0`。
- 此时最小值应回到 `-2`，说明辅助栈同步回退成功。


## 14. 一句话复盘

> 这题最关键的不是双栈本身，而是像 Java 原解那样让最小栈和数据栈逐步同步、长度保持一致。
